In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import  *
import time

In [3]:
spark = SparkSession.builder \
.appName("DataframeWriter Json and Csv") \
.master("local[2]") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/09/28 19:01:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.option("compression","gzip") \
.load("file:///opt/examples/datasets/Hotel_Reviews.csv.gz")

In [5]:
df2 = df.withColumn("Tags", 
                     F.split(F.col("Tags"), ",")
                     .cast(ArrayType(StringType()))) \
.withColumn("Review_Date", F.to_date(F.col("Review_Date"),"M/d/yyyy"))

In [6]:
df2.limit(2).toPandas()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683


In [7]:
df2.printSchema()

root
 |-- Hotel_Address: string (nullable = true)
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: date (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Hotel_Name: string (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Negative_Review: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Positive_Review: string (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- Tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- days_since_review: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lng: string (nullable = true)



# json

In [8]:

start_time = time.time()

df2.write \
.format("json") \
.mode("overwrite") \
.save("file:///tmp/pyspark_output_data/hotel_review_json")


print("----------- %s secs ----------" %(time.time() - start_time))

----------- 23.77483868598938 secs ----------


In [9]:
! ls -lh /tmp/pyspark_output_data/hotel_review_json

total 413M
-rw-r--r--. 1 root root    0 Sep 28 19:02 _SUCCESS
-rw-r--r--. 1 root root 413M Sep 28 19:02 part-00000-d4e51947-2a49-48e0-9d6f-74a68c261e0f-c000.json


# json with compression

In [10]:
start_time = time.time()

df2.write \
.format("json") \
.mode("overwrite") \
.option("compression","gzip") \
.save("file:///tmp/pyspark_output_data/hotel_review_json_compress")


print("----------- %s secs ----------" %(time.time() - start_time))

----------- 35.78486609458923 secs ----------


In [13]:
! ls -lh /tmp/pyspark_output_data/hotel_review_json_compress

total 51M
-rw-r--r--. 1 root root   0 Sep 28 19:03 _SUCCESS
-rw-r--r--. 1 root root 51M Sep 28 19:03 part-00000-de039d7f-08c3-4705-b515-032d82126db3-c000.json.gz


In [14]:
spark.stop()

23/09/28 19:09:17 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-fba05bb9-3548-4be9-9d22-0c70a94834cb. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-fba05bb9-3548-4be9-9d22-0c70a94834cb
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:169)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:113)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1231)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:62)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(Array